# Rule-Based Stylometric Authorship Attribution

This notebook wraps the authorship pipeline into a notebook workflow for the ENG 683 coding project. The project stays symbolic and interpretable by using function-word frequencies, Burrows's Delta, POS trigram profiles, and lexical-richness diagnostics.

## Workflow

1. Load either a local corpus or a built-in NLTK demo corpus.
2. Chunk the texts into comparable samples.
3. Train author profiles from the training chunks.
4. Evaluate held-out chunks with Burrows's Delta as the main attribution score.
5. Inspect ranked candidate authors for each unknown chunk.

In [20]:
from collections import Counter
from pathlib import Path

from authorship_pipeline import (
    SourceDocument,
    ensure_nltk_resources,
    evaluate_model,
    load_local_corpus,
    prepare_chunks,
    split_chunks_by_author,
    train_model,
)

In [21]:
# Choose which corpus to load:
# True = local files under data/<author>/<title>.txt
# False = NLTK Gutenberg demo texts from load_demo_documents()
USE_LOCAL_CORPUS = True
LOCAL_CORPUS_DIR = Path("data")

CHUNK_SIZE = 1500
MIN_CHUNK_SIZE = 900
HOLDOUT_PER_AUTHOR = 1
FUNCTION_WORD_FEATURES = 40
POS_TRIGRAM_FEATURES = 30
TOP_RESULTS = 3

In [22]:
def load_demo_documents():
    from nltk.corpus import gutenberg

    return [
        SourceDocument(
            author="Jane Austen",
            title="Sense and Sensibility",
            text=gutenberg.raw("austen-sense.txt"),
        ),
        SourceDocument(
            author="G. K. Chesterton",
            title="The Man Who Was Thursday",
            text=gutenberg.raw("chesterton-thursday.txt"),
        ),
        SourceDocument(
            author="Maria Edgeworth",
            title="Parents' Assistant",
            text=gutenberg.raw("edgeworth-parents.txt"),
        ),
    ]


def load_project_documents(data_dir=Path("data")):
    data_path = Path(data_dir)
    if not data_path.exists():
        raise FileNotFoundError(f"Data folder was not found: {data_path}")

    documents = []
    for author_dir in sorted(path for path in data_path.iterdir() if path.is_dir()):
        for text_path in sorted(author_dir.glob("*.txt")):
            documents.append(
                SourceDocument(
                    author=author_dir.name,
                    title=text_path.stem,
                    text=text_path.read_text(encoding="utf-8"),
                )
            )

    if not documents:
        raise ValueError("No .txt files found. Use data/<author>/<title>.txt, such as data/GPT/example.txt.")

    return documents


def summarize_chunks(chunks):
    counts = Counter(chunk.author for chunk in chunks)
    return {author: counts[author] for author in sorted(counts)}


def build_result_rows(summary, top_results=3):
    rows = []
    for chunk_score in summary.chunk_scores:
        for rank, result in enumerate(chunk_score.ranking[:top_results], start=1):
            rows.append(
                {
                    "true_author": chunk_score.chunk.author,
                    "title": chunk_score.chunk.title,
                    "chunk_index": chunk_score.chunk.chunk_index,
                    "predicted_author": chunk_score.predicted_author,
                    "rank": rank,
                    "candidate_author": result.author,
                    "delta_score": round(result.delta_score, 4),
                    "pos_similarity": round(result.pos_similarity, 4),
                    "lexical_distance": round(result.lexical_distance, 4),
                }
            )
    return rows

In [23]:
# Preview the local files that will be loaded from data/<author>/<title>.txt.
local_text_files = sorted(LOCAL_CORPUS_DIR.glob("*/*.txt"))

print(f"Found {len(local_text_files)} local text files:")
for text_path in local_text_files:
    author = text_path.parent.name
    title = text_path.stem
    word_count = len(text_path.read_text(encoding="utf-8").split())
    print(f"- {author}: {title} ({word_count:,} words)")

Found 4 local text files:
- Claude: The City That Forgot Its Own Name (36,792 words)
- Claude: The Last Translator of Dreams (35,478 words)
- GPT: The Museum of Unlived Lives (28,009 words)
- GPT: When the Ocean Started Answering (27,713 words)


In [30]:
ensure_nltk_resources(include_gutenberg=not USE_LOCAL_CORPUS)

if USE_LOCAL_CORPUS:
    documents = load_project_documents(LOCAL_CORPUS_DIR)
    corpus_source = f"Local corpus: {LOCAL_CORPUS_DIR}"
else:
    documents = load_demo_documents()
    corpus_source = "NLTK Gutenberg demo corpus"

print(corpus_source)
[(document.author, document.title) for document in documents]

NLTK Gutenberg demo corpus


[('Jane Austen', 'Sense and Sensibility'),
 ('G. K. Chesterton', 'The Man Who Was Thursday'),
 ('Maria Edgeworth', "Parents' Assistant")]

In [31]:
chunks = prepare_chunks(
    documents,
    chunk_size=CHUNK_SIZE,
    min_chunk_size=MIN_CHUNK_SIZE,
)

print("Prepared chunks by author:")
summarize_chunks(chunks)

Prepared chunks by author:


{'G. K. Chesterton': 39, 'Jane Austen': 81, 'Maria Edgeworth': 114}

In [32]:
training_chunks, test_chunks = split_chunks_by_author(
    chunks,
    holdout_per_author=HOLDOUT_PER_AUTHOR,
)

model = train_model(
    training_chunks,
    function_word_features=FUNCTION_WORD_FEATURES,
    pos_trigram_features=POS_TRIGRAM_FEATURES,
)

print(f"Training chunks: {len(training_chunks)}")
print(f"Test chunks: {len(test_chunks)}")
print(f"Function-word features kept: {len(model.function_words)}")
print(f"POS trigram features kept: {len(model.pos_trigrams)}")
model.function_words[:20]

Training chunks: 231
Test chunks: 3
Function-word features kept: 40
POS trigram features kept: 30


('the',
 'to',
 'and',
 'of',
 'a',
 'i',
 'in',
 'he',
 'you',
 'was',
 'it',
 'that',
 'her',
 'his',
 'for',
 'not',
 'as',
 'she',
 'with',
 'be')

In [33]:
summary = evaluate_model(model, test_chunks)
print(
    f"Holdout accuracy: {summary.correct_predictions}/{summary.total_chunks} = {summary.accuracy:.2%}"
)

Holdout accuracy: 3/3 = 100.00%


In [34]:
results = build_result_rows(summary, top_results=TOP_RESULTS)
results

[{'true_author': 'G. K. Chesterton',
  'title': 'The Man Who Was Thursday',
  'chunk_index': 38,
  'predicted_author': 'G. K. Chesterton',
  'rank': 1,
  'candidate_author': 'G. K. Chesterton',
  'delta_score': 0.7242,
  'pos_similarity': 0.9667,
  'lexical_distance': 1.6187},
 {'true_author': 'G. K. Chesterton',
  'title': 'The Man Who Was Thursday',
  'chunk_index': 38,
  'predicted_author': 'G. K. Chesterton',
  'rank': 2,
  'candidate_author': 'Maria Edgeworth',
  'delta_score': 0.7853,
  'pos_similarity': 0.9363,
  'lexical_distance': 3.833},
 {'true_author': 'G. K. Chesterton',
  'title': 'The Man Who Was Thursday',
  'chunk_index': 38,
  'predicted_author': 'G. K. Chesterton',
  'rank': 3,
  'candidate_author': 'Jane Austen',
  'delta_score': 0.9655,
  'pos_similarity': 0.908,
  'lexical_distance': 11.4017},
 {'true_author': 'Jane Austen',
  'title': 'Sense and Sensibility',
  'chunk_index': 80,
  'predicted_author': 'Jane Austen',
  'rank': 1,
  'candidate_author': 'Jane Austen

## Final Project Notes

This notebook can load either corpus source. In Cell 4, set `USE_LOCAL_CORPUS = True` to use your own files under `data/<author>/<title>.txt`, or set `USE_LOCAL_CORPUS = False` to use the built-in NLTK Gutenberg demo texts. The strongest version of this project uses multiple texts per author in the same general genre and period, because authorship attribution is sensitive to topic, genre, and text length.

In [9]:
# Optional export cell
# Uncomment these lines if you want to save the ranked results to disk.
# from json import dumps
# results_path = Path("data/holdout_results.json")
# results_path.write_text(dumps(results, indent=2), encoding="utf-8")
# results_path

In [35]:
# Compare the local Claude/GPT corpus with the NLTK Gutenberg demo corpus.
from pprint import pprint


def summarize_experiment(label, documents):
    experiment_chunks = prepare_chunks(
        documents,
        chunk_size=CHUNK_SIZE,
        min_chunk_size=MIN_CHUNK_SIZE,
    )
    experiment_training, experiment_test = split_chunks_by_author(
        experiment_chunks,
        holdout_per_author=HOLDOUT_PER_AUTHOR,
    )
    experiment_model = train_model(
        experiment_training,
        function_word_features=FUNCTION_WORD_FEATURES,
        pos_trigram_features=POS_TRIGRAM_FEATURES,
    )
    experiment_summary = evaluate_model(experiment_model, experiment_test)

    return {
        "corpus": label,
        "documents": [(document.author, document.title) for document in documents],
        "chunks_by_author": summarize_chunks(experiment_chunks),
        "training_chunks": len(experiment_training),
        "test_chunks": len(experiment_test),
        "function_word_features_kept": len(experiment_model.function_words),
        "pos_trigram_features_kept": len(experiment_model.pos_trigrams),
        "accuracy": f"{experiment_summary.correct_predictions}/{experiment_summary.total_chunks} = {experiment_summary.accuracy:.2%}",
        "top_results": build_result_rows(experiment_summary, top_results=TOP_RESULTS),
    }


ensure_nltk_resources(include_gutenberg=True)
local_summary = summarize_experiment("Local Claude/GPT generated fiction", load_project_documents(LOCAL_CORPUS_DIR))
gutenberg_summary = summarize_experiment("NLTK Gutenberg demo corpus", load_demo_documents())

pprint({
    "local": local_summary,
    "gutenberg": gutenberg_summary,
}, sort_dicts=False)

{'local': {'corpus': 'Local Claude/GPT generated fiction',
           'documents': [('Claude', 'The City That Forgot Its Own Name'),
                         ('Claude', 'The Last Translator of Dreams'),
                         ('GPT', 'The Museum of Unlived Lives'),
                         ('GPT', 'When the Ocean Started Answering')],
           'chunks_by_author': {'Claude': 49, 'GPT': 38},
           'training_chunks': 85,
           'test_chunks': 2,
           'function_word_features_kept': 40,
           'pos_trigram_features_kept': 30,
           'accuracy': '2/2 = 100.00%',
           'top_results': [{'true_author': 'Claude',
                            'title': 'The Last Translator of Dreams',
                            'chunk_index': 23,
                            'predicted_author': 'Claude',
                            'rank': 1,
                            'candidate_author': 'Claude',
                            'delta_score': 1.2357,
                            'pos_s